# Building a Decorator


To see how this works under the hood, let's build a practical decorator: a simple execution timer.

A standard decorator requires three layers:

* The Decorator Function: This takes the original function as an argument.

* The Wrapper Function: This is the inner function (a closure) that actually intercepts the call, executes custom logic, calls the original function, and returns the result.

* The Return: The decorator returns the wrapper function object (without executing it).

In [6]:
import time

# 1. The Decorator
def timer(func):
    
    # 2. The Wrapper (uses *args and **kwargs to accept ANY arguments)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        
        # We execute the original function here
        result = func(*args, **kwargs)
        
        end_time = time.perf_counter()
        print(f"Function '{func.__name__}' took {end_time - start_time:.4f} seconds")
        
        # We must return the result so the function behaves normally
        return result
        
    # 3. Return the wrapper object
    return wrapper

# Applying the decorator
@timer
def heavy_computation(n):
    return sum(i * i for i in range(n))

# When we call heavy_computation, we are actually calling 'wrapper'
heavy_computation(1000000)
# Output: Function 'heavy_computation' took 0.0521 seconds

Function 'heavy_computation' took 0.2117 seconds


333332833333500000

### The Metadata Problem (functools.wraps)
There is a subtle bug in the pattern above. Because we replaced heavy_computation with wrapper, we lost all of the original function's metadata—its name, its docstring, and its module.

In [7]:
print(heavy_computation.__name__)  
# Output: 'wrapper' (Oops! We lost the name 'heavy_computation')

wrapper


This makes debugging a nightmare, and it breaks tools that rely on introspection (like testing frameworks or API documentation generators like FastAPI/Swagger).

To fix this, Python provides a built-in decorator for decorators, called functools.wraps. It copies the metadata from the original function to the wrapper.

In [8]:
from functools import wraps
import time

def timer(func):
    @wraps(func)  # <-- This copies the metadata from 'func' onto 'wrapper'
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Function '{func.__name__}' took {end_time - start_time:.4f} seconds")
        return result
    return wrapper

@timer
def heavy_computation(n):
    """Calculates the sum of squares up to n."""
    return sum(i * i for i in range(n))

print(heavy_computation.__name__) 
# Output: 'heavy_computation' (Metadata preserved!)

print(heavy_computation.__doc__)
# Output: 'Calculates the sum of squares up to n.'

heavy_computation
Calculates the sum of squares up to n.


### Decorators with Parameters (The 3-Layer Pattern)
How does something like @lru_cache(maxsize=128) work? If the decorator needs to accept arguments, you have to add an outer layer. The outermost function takes the arguments, and returns the actual decorator.

In [9]:
def repeat(num_times):
    # This is the actual decorator
    def decorator_repeat(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(num_times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    # Return the decorator
    return decorator_repeat

@repeat(num_times=3)
def greet(name):
    print(f"Hello {name}")

greet("Shubham")
# Output:
# Hello Shubham
# Hello Shubham
# Hello Shubham

Hello Shubham
Hello Shubham
Hello Shubham
